In [ ]:
# SOURCE OF INPIRATION
# Shows how construct the training set for LLaMA format

import jsonlines

# jsonl input and output paths
jsonl_input_paths = ["OPL-QA.jsonl", "OPL-Language.jsonl", "OPL-Examples.jsonl"]
jsonl_output_path = "train.jsonl"

# Open the output JSONL file
with open(jsonl_output_path, "w") as output_file:
    json_writer = jsonlines.Writer(output_file)

    for jsonl_input_path in jsonl_input_paths:
        with open(jsonl_input_path, "r") as input_file:
            json_reader = jsonlines.Reader(input_file)

            for entry in json_reader:
                try:
                    # Combine prompt and completion into the desired format
                    formatted_text = f"<s>[INST] {entry['prompt']} [/INST] {entry['completion']} </s>"

                    # Write the formatted text as a JSONL line
                    json_writer.write({"text": formatted_text})
                except Exception as e:
                    print(f"Error parsing JSON in {jsonl_input_path}:", entry, e)

# 1. Sampling original dataset

In [212]:
import json
import numpy as np
import math

def sampling_records(source_jsonl, sample_ratio = 0.01, target_jsonl=""):
    with open(source_jsonl) as f:
        records = [json.loads(x) for x in f]

    count = len(records)
    print('total records in source file:', count)
    sample_count = math.ceil(count * sample_ratio)

    sample_indexes = np.random.choice(count,sample_count)

    sample_records = []
    for sample_index in sample_indexes:
        sample_record = records[sample_index]
        #print(type(sample_record))
        sample_records.append(sample_record)

    assert len(sample_records) == sample_count

    if target_jsonl != "":
        with open(target_jsonl, "w") as f:
            for record in sample_records:
                f.write(json.dumps(record) + "\n")

        print("Sampled {} records from {} original records.".format(sample_count, count))
    else:
        print("Sampled {} records from {} original records.".format(sample_count, count))
        return sample_records

### 1.1 Test sampling to file

In [224]:
source_jsonl = "/Users/beto/Documents/Projects/medLLM/oia_arranged.jsonl"
target_jsonl = "/Users/beto/Documents/Projects/medLLM/sampling.jsonl"
sample_ratio = 0.001


In [225]:

sampling_records(source_jsonl, sample_ratio=sample_ratio, target_jsonl=target_jsonl)


total records in source file: 366728
Sampled 367 records from 366728 original records.


### 1.2 Test sampling to memory

In [226]:
sampled_records = sampling_records(source_jsonl)

total records in source file: 366728
Sampled 3668 records from 366728 original records.


# Testing Regular Expression

In [227]:
import re

s = 'Make the World a *Better Place*'
pattern = r'\*(.*?)\*'
replacement = r'<b>\1<\\b>'
html = re.sub(pattern, replacement, s)

print(html)

Make the World a <b>Better Place<\b>


In [228]:
import re

# fiven a string with sequences IDs, they are listed
re.findall( r'[A][0-9]{6}', 'this is a long string with A050343, several patternst o find, for example: A084673 and A072683')

['A050343', 'A084673', 'A072683']

# Creating Questions

In [229]:
len(sampled_records)

3668

### Replacing IDs with actual sequences

In [262]:
import re
import search_engine

# if a Name exists in the target text, it is subsituted by the real sequence
def replace_seq_id_by_real_sequence(target_string: str):
    references = []
    if len(target_string) > 0:
        references = re.findall( r'[A][0-9]{6}', target_string)
        # print(references)
        for r in references:
            seq_number = int(r.strip("A"))
            real_sequence = search_engine.get_sequence_terms_by_number(seq_number)
            #print('real sequence: ', real_sequence)
            target_string = target_string.replace(r, "[" + real_sequence + "]")
    return target_string

In [263]:
# Deliver the original with the substitution of the id of a sequence by the actual sequence
replace_seq_id_by_real_sequence('First column of the triangle described in A082200.')

'First column of the triangle described in [1,4,9,25,49,8,6,121,15,77,35,169,16,221,10,12,187,21,209,27,91,65,361,20,289,32,55,18,14,33,161,39,133,299,119,95,85,247,34,125,22,45,44,69,26,24,203,81,217,323,259,377,155,287,51,115,143,38,145,36,205,46,57,52].'

In [280]:
# If a record has defined thekey "name" and it contains the code in Mathematica/Maple, 
# design the questions for training

def question_by_NAME(dict_record: dict):
    #print("before: ",dict_record['name'] )
    info_name = replace_seq_id_by_real_sequence(dict_record['name'])
    #print("after: ", info_name)
    mathematica_code = dict_record['mathematica']
    maple_code = dict_record['maple']
    questions = []
    #print(mathematica_code)
    if len(info_name) > 0 and len(mathematica_code) > 0:
        formatted_text = "<s>[INST] The code in Mathematica for the {} is: [/INST] {} </s>".format(dict_record['name'],dict_record['mathematica'] )
        #print(formatted_text)
        questions.append(str(formatted_text))
    if len(info_name) > 0 and len(maple_code) > 0:
        formatted_text = "<s>[INST] The code in Maple for the {} is: [/INST] {} </s>".format(dict_record['name'], dict_record['maple'])
        #print(formatted_text)
        questions.append(str(formatted_text))
    return questions


# If a record has mathematica or Maple code, attach it to the sequence itself
def question_by_DATA(dict_record: dict):
    mathematica_code = dict_record['mathematica']
    maple_code = dict_record['maple']
    questions = []
    if len(mathematica_code) > 0:
        formatted_text = f"<s>[INST] The code in Mathematica for the sequence {dict_record['data']} is: [/INST] {dict_record['mathematica']} </s>"
        # print(formatted_text)
        questions.append(formatted_text)
    if len(maple_code) > 0:
        formatted_text = f"<s>[INST] The code in Maple for the sequence {dict_record['data']} is: [/INST] {dict_record['maple']} </s>"
        # print(formatted_text)
        questions.append(formatted_text)
    return questions


# by FORMULA
def question_by_FORMULA(dict_record: dict):
    # print("processing: ")
    # print(dict_record['formula'])
    if isinstance(dict_record['formula'], list):
        text = ' '.join(dict_record['formula'])
    else:
        text = dict_record['formula']
    formula_text = replace_seq_id_by_real_sequence(text)
    # print(formula_text)
    mathematica_code = dict_record['mathematica']
    maple_code = dict_record['maple']
    questions = []
    if len(formula_text) > 0 and len(mathematica_code) > 0:
        formatted_text = f"<s>[INST] The code in Mathematica for the formula {dict_record['formula']} is: [/INST] {dict_record['mathematica']} </s>"
        # print(formatted_text)
        questions.append(formatted_text)
    if len(formula_text) > 0 and len(maple_code) > 0:
        formatted_text = f"<s>[INST] The code in Maple for the formula {dict_record['formula']} is: [/INST] {dict_record['maple']} </s>"
        # print(formatted_text)
        questions.append(formatted_text)
    return questions

def create_questions(dic_record:dict):
    name_questions = question_by_NAME(dic_record)
    data_questions = question_by_DATA(dic_record)
    formula_questions = question_by_FORMULA(dic_record)
    out = []
    for q in name_questions:
        out.append(q)
    for q in data_questions:
        out.append(q)
    for q in formula_questions:
        out.append(q)
    return out

# testing creation of questions

### Independet test

In [281]:
from random import randrange

# Independent test
name_qs = []
while len(name_qs) == 0:
    register_1 = randrange(len(sampled_records)-1)
    name_qs = question_by_NAME(sampled_records[register_1])
print("register: ", register_1)

data_qs = []
while len(data_qs) == 0:
    register_2 = randrange(len(sampled_records)-1)
    data_qs = question_by_DATA(sampled_records[register_2])
print("register: ",register_2)

formula_qs = []
while len(formula_qs) == 0:
    register_3 = randrange(len(sampled_records)-1)
    formula_qs = question_by_FORMULA(sampled_records[register_3])
print("register: ",register_3)

print("\n")
print("quest by name ({}):".format(register_1))
print(name_qs)
print("quest by data ({}): ".format(register_2))
print(data_qs)
print("quest by formula ({}): ".format(register_3))
print(formula_qs)

register:  2
register:  21
register:  2


quest by name (2):
["<s>[INST] The code in Mathematica for the Number of aperiodic necklaces of n beads of 11 colors. is: [/INST] ['f[d_]:=MoebiusMu[d] 11^(n/d)/n; a[n_]:=Total[f/@Divisors[n]]; a[0]=1; Table[a[n], {n, 1, 30}] (* _Vincenzo Librandi_, Oct 14 2017 *)'] </s>", "<s>[INST] The code in Maple for the Number of aperiodic necklaces of n beads of 11 colors. is: [/INST] ['f:= (n,p) -> add(numtheory:-mobius(d)*p^(n/d),d=numtheory:-divisors(n))/n:', 'seq(f(n,11), n=1..100); # _Robert Israel_, Jan 07 2015'] </s>"]
quest by data (21): 
["<s>[INST] The code in Mathematica for the sequence 3,20,142,1070,8482,69950,593722,5141030,45116962,399451310,3557016202,31792684790,284850459442,2556147225470,22961260134682,206391834657350,1855993886649922,16694871298564430,150200009950933162 is: [/INST] ['Table[5^n + 6^n + 9^n, {n, 0, 20}]'] </s>"]
quest by formula (2): 
['<s>[INST] The code in Mathematica for the formula [\'"CHK" (necklace, identity, unlab

### Serial test

In [282]:
print(" *** length of sampled records:")
print(len(sampled_records))
# Single test of creating questions
print("/n")
print(" *** creating questions of one single record:")


from random import randrange
qstns = []
while len(qstns) == 0:
    index = randrange(len(sampled_records)-1)
    qstns = create_questions(sampled_records[index])
print("record index: ", index)
for q in qstns:
    print(q)

 *** length of sampled records:
37
/n
 *** creating questions of one single record:
record index:  21
<s>[INST] The code in Mathematica for the a(n) = 5^n + 6^n + 9^n. is: [/INST] ['Table[5^n + 6^n + 9^n, {n, 0, 20}]'] </s>
<s>[INST] The code in Mathematica for the sequence 3,20,142,1070,8482,69950,593722,5141030,45116962,399451310,3557016202,31792684790,284850459442,2556147225470,22961260134682,206391834657350,1855993886649922,16694871298564430,150200009950933162 is: [/INST] ['Table[5^n + 6^n + 9^n, {n, 0, 20}]'] </s>
<s>[INST] The code in Mathematica for the formula ['From _Mohammad K. Azarian_, Dec 30 2008: (Start)', 'G.f.: 1/(1-5*x) + 1/(1-6*x) + 1/(1-9*x).', 'E.g.f.: e^(5*x) + e^(6*x) + e^(9*x). (End)'] is: [/INST] ['Table[5^n + 6^n + 9^n, {n, 0, 20}]'] </s>


# Generating training dataset and sampling

In [288]:
import jsonlines

# jsonl input and output paths
jsonl_input_paths = ["oia_arranged.jsonl"]
jsonl_output_path = "train.jsonl"

sampling_mode = "random" # "random" or "sequential"
source_jsonl = "/Users/beto/Documents/Projects/medLLM/oia_arranged.jsonl"
target_jsonl = "/Users/beto/Documents/Projects/medLLM/sampling.jsonl"
sample_ratio = 1

# Open the output JSONL file
with open(jsonl_output_path, "w") as output_file:
    json_writer = jsonlines.Writer(output_file)

    for jsonl_input_path in jsonl_input_paths:
        with open(jsonl_input_path, "r") as input_file:
            sampled_records = sampling_records(source_jsonl, sample_ratio=sample_ratio)#, target_jsonl=target_jsonl) # decomment if you wanna save the file as sampling
            num_questions = 0
            for one_record in sampled_records:
                try:
                    # Combine prompt and completion into the desired format
                    questions = create_questions(one_record)

                    for q in questions:
                        num_questions = num_questions + 1
                        #print(str(num_questions), ' :', q)

                        # Write the formatted text as a JSONL line
                        json_writer.write({"text": q})
                except Exception as e:
                    print(f"Error parsing JSON in {jsonl_input_path}:", one_record, e)
                    
    print("Number of questions: ", num_questions)
    print("Resulting file: ", jsonl_output_path)


total records in source file: 366728
Sampled 366728 records from 366728 original records.
Number of questions:  634470
Resulting file:  train.jsonl


In [289]:
# from itertools import islice
# from oeis_sequences.OEISsequences import A114274_gen
# print(list(islice(A114274_gen(),10)))